In [ ]:
#DailyChallenge

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import pearsonr, f_oneway, skew, kurtosis
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configuration du style des graphiques
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Chargement des données
df = pd.read_csv(r'C:\Users\kmeso\Desktop\TTA_KOUAME_ASSE_SOKRY_JOSEPH\WEEK03\CSV\train.csv')

# Exploration initiale
print("="*50)
print("APERÇU DES DONNÉES")
print("="*50)
print(f"Shape du dataset: {df.shape}")
print(f"\nNombre de lignes: {df.shape[0]}")
print(f"Nombre de colonnes: {df.shape[1]}")

print("\n" + "="*50)
print("PREMIÈRES LIGNES")
print("="*50)
df.head()

# Informations sur les colonnes
print("\n" + "="*50)
print("INFORMATIONS SUR LES COLONNES")
print("="*50)
df.info()

# Description des colonnes et types
print("\n" + "="*50)
print("TYPES DE DONNÉES")
print("="*50)
type_df = pd.DataFrame({
    'Colonne': df.columns,
    'Type': df.dtypes.values,
    'Valeurs non-nulles': df.count().values,
    '% valeurs manquantes': (df.isnull().sum() / len(df) * 100).values
})
type_df

# %%
# Statistiques descriptives de base
print("\n" + "="*50)
print("STATISTIQUES DESCRIPTIVES")
print("="*50)
df.describe()

# %%
# Vérification des valeurs manquantes
print("\n" + "="*50)
print("VALEURS MANQUANTES")
print("="*50)
missing_values = df.isnull().sum()
missing_percentage = (missing_values / len(df)) * 100
missing_df = pd.DataFrame({
    'Valeurs manquantes': missing_values,
    'Pourcentage': missing_percentage
})
missing_df[missing_df['Valeurs manquantes'] > 0]

# %% [markdown]
# ## 2. Nettoyage et Prétraitement des Données

# %%
# Supprimer les colonnes avec trop de valeurs manquantes (>50%)
threshold = 0.5
cols_to_drop = missing_df[missing_df['Pourcentage'] > threshold*100].index
df_clean = df.drop(columns=cols_to_drop)
print(f"Colonnes supprimées: {list(cols_to_drop)}")
print(f"Nouveau shape: {df_clean.shape}")

# %%
# Imputer les valeurs manquantes restantes
from sklearn.impute import SimpleImputer

# Séparer les colonnes numériques et catégorielles
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
categorical_cols = df_clean.select_dtypes(include=['object']).columns

# Imputer les valeurs numériques avec la médiane
if len(numeric_cols) > 0:
    imputer_num = SimpleImputer(strategy='median')
    df_clean[numeric_cols] = imputer_num.fit_transform(df_clean[numeric_cols])

# Imputer les valeurs catégorielles avec le mode
if len(categorical_cols) > 0:
    imputer_cat = SimpleImputer(strategy='most_frequent')
    df_clean[categorical_cols] = imputer_cat.fit_transform(df_clean[categorical_cols])

print("Valeurs manquantes imputées")

# Vérification finale des valeurs manquantes
print("\nVérification finale:")
print(df_clean.isnull().sum().sum(), "valeurs manquantes restantes")

# Transformation des données catégorielles
# Identifier la colonne cible 'price_range'
target_col = 'price_range' if 'price_range' in df_clean.columns else None

# Encoder les variables catégorielles
from sklearn.preprocessing import LabelEncoder

df_encoded = df_clean.copy()
label_encoders = {}

for col in categorical_cols:
    if col != target_col:
        le = LabelEncoder()
        df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))
        label_encoders[col] = le
        print(f"Encodé: {col}")

print("\nVariables catégorielles transformées")

# ## 3. Analyse Statistique avec NumPy et SciPy

# Séparer les features et la target
if target_col:
    X = df_encoded.drop(columns=[target_col])
    y = df_encoded[target_col]
else:
    X = df_encoded
    y = None
    print("Colonne 'price_range' non trouvée")

# Analyse statistique détaillée de chaque caractéristique
print("="*60)
print("ANALYSE STATISTIQUE DÉTAILLÉE DES CARACTÉRISTIQUES")
print("="*60)

stats_results = []

for col in X.columns:
    data = X[col].dropna()
    
    # Mesures de tendance centrale
    mean_val = np.mean(data)
    median_val = np.median(data)
    try:
        mode_val = stats.mode(data, keepdims=True)[0][0]
    except:
        mode_val = np.nan
    
    # Mesures de variabilité
    range_val = np.ptp(data)
    variance_val = np.var(data)
    std_val = np.std(data)
    
    # Forme de distribution
    skewness = skew(data)
    kurt = kurtosis(data)
    
    # Percentiles
    percentiles = np.percentile(data, [25, 50, 75])
    
    stats_results.append({
        'Feature': col,
        'Mean': mean_val,
        'Median': median_val,
        'Mode': mode_val,
        'Range': range_val,
        'Variance': variance_val,
        'Std': std_val,
        'Skewness': skewness,
        'Kurtosis': kurt,
        'Q1': percentiles[0],
        'Q3': percentiles[2]
    })

stats_df = pd.DataFrame(stats_results)
stats_df

# Tests d'hypothèse par gamme de prix
if target_col and y is not None:
    print("\n" + "="*60)
    print("TESTS D'HYPOTHÈSE: DIFFÉRENCES ENTRE GAMMES DE PRIX")
    print("="*60)
    
    # ANOVA pour chaque feature
    anova_results = []
    
    for col in X.columns:
        # Créer des groupes par gamme de prix
        groups = [X[col][y == price] for price in sorted(y.unique())]
        
        # Exécuter ANOVA
        try:
            f_stat, p_value = f_oneway(*groups)
            anova_results.append({
                'Feature': col,
                'F-statistic': f_stat,
                'P-value': p_value,
                'Significatif (p<0.05)': p_value < 0.05
            })
        except:
            anova_results.append({
                'Feature': col,
                'F-statistic': np.nan,
                'P-value': np.nan,
                'Significatif (p<0.05)': False
            })
    
    anova_df = pd.DataFrame(anova_results)
    anova_df = anova_df.sort_values('P-value')
    print("\nFeatures les plus significatives pour la classification des prix:")
    print(anova_df.head(10))

# Analyse de corrélation avec SciPy
print("\n" + "="*60)
print("ANALYSE DE CORRÉLATION")
print("="*60)

correlations = []

for col in X.columns:
    if target_col and y is not None:
        # Corrélation avec la target
        corr, p_value = pearsonr(X[col].fillna(0), y.fillna(0))
        correlations.append({
            'Feature': col,
            'Target': target_col,
            'Correlation': corr,
            'P-value': p_value,
            'Significatif': p_value < 0.05
        })

corr_df = pd.DataFrame(correlations)
corr_df = corr_df.reindex(corr_df['Correlation'].abs().sort_values(ascending=False).index)
print("\nCorrélations avec la variable cible:")
print(corr_df.head(10))

# %% [markdown]
# ## 4. Visualisation des Données avec Matplotlib

# %%
# Configuration des sous-plots
fig = plt.figure(figsize=(20, 15))

# 1. Distribution des prix
if target_col and y is not None:
    ax1 = fig.add_subplot(3, 3, 1)
    price_counts = y.value_counts().sort_index()
    ax1.bar(price_counts.index, price_counts.values, color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4'])
    ax1.set_xlabel('Gamme de Prix')
    ax1.set_ylabel('Nombre de Téléphones')
    ax1.set_title('Distribution des Gammes de Prix', fontsize=12, fontweight='bold')
    ax1.set_xticks([0, 1, 2, 3])
    ax1.set_xticklabels(['Bas', 'Moyen-bas', 'Moyen-haut', 'Haut'])

# 2. Top 10 des corrélations
ax2 = fig.add_subplot(3, 3, 2)
top_features = corr_df.head(10)
colors = ['red' if x < 0 else 'green' for x in top_features['Correlation']]
ax2.barh(top_features['Feature'], top_features['Correlation'], color=colors)
ax2.set_xlabel('Corrélation')
ax2.set_title('Top 10 des Features Corrélées avec le Prix', fontsize=12, fontweight='bold')
ax2.axvline(x=0, color='black', linestyle='-', linewidth=0.5)

# 3. Histogrammes des features importantes
ax3 = fig.add_subplot(3, 3, 3)
important_feature = top_features.iloc[0]['Feature'] if len(top_features) > 0 else X.columns[0]
ax3.hist(X[important_feature], bins=30, color='#4ECDC4', edgecolor='black', alpha=0.7)
ax3.set_xlabel(important_feature)
ax3.set_ylabel('Fréquence')
ax3.set_title(f'Distribution de {important_feature}', fontsize=12, fontweight='bold')
ax3.axvline(X[important_feature].mean(), color='red', linestyle='--', label=f'Moyenne: {X[important_feature].mean():.2f}')
ax3.axvline(X[important_feature].median(), color='green', linestyle='--', label=f'Médiane: {X[important_feature].median():.2f}')
ax3.legend()

# 4. Box plots par gamme de prix
if target_col and y is not None:
    ax4 = fig.add_subplot(3, 3, 4)
    top_feature = top_features.iloc[0]['Feature'] if len(top_features) > 0 else X.columns[0]
    data_by_price = [X[top_feature][y == price] for price in sorted(y.unique())]
    bp = ax4.boxplot(data_by_price, labels=['Bas', 'Moyen-bas', 'Moyen-haut', 'Haut'], patch_artist=True)
    for box in bp['boxes']:
        box.set_facecolor('#FF6B6B')
        box.set_alpha(0.7)
    ax4.set_xlabel('Gamme de Prix')
    ax4.set_ylabel(top_feature)
    ax4.set_title(f'{top_feature} par Gamme de Prix', fontsize=12, fontweight='bold')
    ax4.grid(True, alpha=0.3)

# 5. Scatter plot
ax5 = fig.add_subplot(3, 3, 5)
if len(top_features) > 1:
    feature1 = top_features.iloc[0]['Feature']
    feature2 = top_features.iloc[1]['Feature']
    scatter = ax5.scatter(X[feature1], X[feature2], c=y if target_col else 'blue', 
                         cmap='viridis', alpha=0.6, s=50)
    ax5.set_xlabel(feature1)
    ax5.set_ylabel(feature2)
    ax5.set_title(f'{feature1} vs {feature2} (couleur = prix)', fontsize=12, fontweight='bold')
    if target_col:
        plt.colorbar(scatter, ax=ax5, label='Gamme de Prix')

# 6. Heatmap de corrélation
ax6 = fig.add_subplot(3, 3, 6)
# Sélectionner les 15 premières features
top_corr_features = corr_df.head(15)['Feature'].tolist()
if target_col:
    top_corr_features.append(target_col)
corr_matrix = df_encoded[top_corr_features].corr()
im = ax6.imshow(corr_matrix, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
ax6.set_xticks(range(len(top_corr_features)))
ax6.set_yticks(range(len(top_corr_features)))
ax6.set_xticklabels(top_corr_features, rotation=45, ha='right', fontsize=8)
ax6.set_yticklabels(top_corr_features, fontsize=8)
ax6.set_title('Matrice de Corrélation (Top 15 Features)', fontsize=12, fontweight='bold')
plt.colorbar(im, ax=ax6)

# 7. Distribution des skewness
ax7 = fig.add_subplot(3, 3, 7)
skew_values = stats_df['Skewness'].dropna()
colors = ['red' if x > 1 or x < -1 else 'orange' if x > 0.5 or x < -0.5 else 'green' for x in skew_values]
ax7.bar(range(len(skew_values)), skew_values, color=colors)
ax7.set_xlabel('Features')
ax7.set_ylabel('Skewness')
ax7.set_title('Distribution de l\'Asymétrie par Feature', fontsize=12, fontweight='bold')
ax7.axhline(y=0.5, color='orange', linestyle='--', alpha=0.7, label='Seuil modéré')
ax7.axhline(y=1, color='red', linestyle='--', alpha=0.7, label='Seuil élevé')
ax7.axhline(y=-0.5, color='orange', linestyle='--', alpha=0.7)
ax7.axhline(y=-1, color='red', linestyle='--', alpha=0.7)
ax7.legend()
ax7.set_xticks([])

# 8. Écart-type des features
ax8 = fig.add_subplot(3, 3, 8)
std_sorted = stats_df.nlargest(15, 'Std')
ax8.barh(std_sorted['Feature'], std_sorted['Std'], color='#96CEB4')
ax8.set_xlabel('Écart-type')
ax8.set_title('Top 15 Features par Variabilité', fontsize=12, fontweight='bold')

# 9. Analyse ANOVA - P-values
if target_col and y is not None:
    ax9 = fig.add_subplot(3, 3, 9)
    significant = anova_df[anova_df['Significatif (p<0.05)'] == True].head(10)
    if len(significant) > 0:
        ax9.barh(significant['Feature'], -np.log10(significant['P-value']), color='#45B7D1')
        ax9.set_xlabel('-log10(P-value)')
        ax9.set_title('Significativité Statistique (ANOVA)', fontsize=12, fontweight='bold')
        ax9.axvline(x=-np.log10(0.05), color='red', linestyle='--', label='Seuil p=0.05')
        ax9.legend()

plt.tight_layout()
plt.show()

# Visualisations supplémentaires

# Box plots multiples pour les meilleures features
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

if target_col and y is not None:
    top_6_features = top_features.head(6)['Feature'].tolist()
    
    for idx, feature in enumerate(top_6_features):
        data_by_price = [X[feature][y == price] for price in sorted(y.unique())]
        bp = axes[idx].boxplot(data_by_price, labels=['Bas', 'Moyen-bas', 'Moyen-haut', 'Haut'], 
                               patch_artist=True)
        for box in bp['boxes']:
            box.set_facecolor('#4ECDC4')
            box.set_alpha(0.7)
        axes[idx].set_xlabel('Gamme de Prix')
        axes[idx].set_ylabel(feature)
        axes[idx].set_title(f'{feature}', fontsize=10, fontweight='bold')
        axes[idx].grid(True, alpha=0.3)
    
    # Cacher le dernier subplot si nécessaire
    if len(top_6_features) < 6:
        axes[-1].set_visible(False)

plt.suptitle('Distribution des Top Features par Gamme de Prix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ## 5. Synthèse des Insights et Conclusion
print("="*70)
print("SYNTHÈSE DES ANALYSES ET CONCLUSIONS")
print("="*70)

# 1. Principaux facteurs déterminants
print("\n1. PRINCIPAUX FACTEURS DÉTERMINANTS DU PRIX:")
print("-"*50)
top_5_features = corr_df.head(5)
for idx, row in top_5_features.iterrows():
    impact = "positif" if row['Correlation'] > 0 else "négatif"
    strength = "fort" if abs(row['Correlation']) > 0.3 else "modéré" if abs(row['Correlation']) > 0.1 else "faible"
    print(f"  • {row['Feature']}: Corrélation {impact} {strength} ({row['Correlation']:.3f})")

# 2. Distribution des données
print("\n2. CARACTÉRISTIQUES DES DISTRIBUTIONS:")
print("-"*50)
high_skew = stats_df[abs(stats_df['Skewness']) > 1]
high_kurt = stats_df[stats_df['Kurtosis'] > 3]

print(f"  • {len(high_skew)} features présentent une asymétrie forte (|skewness| > 1)")
print(f"  • {len(high_kurt)} features présentent un kurtosis élevé (>3) indiquant des queues épaisses")

# 3. Résultats des tests statistiques
if target_col and y is not None:
    print("\n3. TESTS STATISTIQUES (ANOVA):")
    print("-"*50)
    significant_features = anova_df[anova_df['Significatif (p<0.05)'] == True]
    print(f"  • {len(significant_features)} features sont statistiquement significatives")
    print(f"  • La feature la plus discriminante: {significant_features.iloc[0]['Feature']} (p={significant_features.iloc[0]['P-value']:.2e})")

# 4. Insights clés
print("\n4. INSIGHTS CLÉS:")
print("-"*50)
print("  a) Les caractéristiques techniques ont l'impact le plus fort sur le prix")
print("  b) Certaines features montrent une séparation claire entre les gammes de prix")
print("  c) La variabilité des données est importante pour certaines caractéristiques clés")

# 5. Recommandations
print("\n5. RECOMMANDATIONS POUR LA MODÉLISATION:")
print("-"*50)
print("  ✓ Prioriser les features avec la plus forte corrélation")
print("  ✓ Considérer la transformation des features asymétriques")
print("  ✓ Les tests ANOVA confirment la pertinence des features sélectionnées")

# Export des résultats
results_summary = {
    'statistiques_features': stats_df,
    'correlations_prix': corr_df if target_col else None,
    'anova_results': anova_df if target_col else None
}

# Sauvegarder les résultats en CSV
stats_df.to_csv('analyse_statistique_features.csv', index=False)
if target_col:
    corr_df.to_csv('correlations_prix.csv', index=False)
    anova_df.to_csv('anova_resultats.csv', index=False)

print("\nRésultats exportés dans des fichiers CSV")
print("   - analyse_statistique_features.csv")
if target_col:
    print("   - correlations_prix.csv")
    print("   - anova_resultats.csv")

print("\n" + "="*70)
print("ANALYSE TERMINÉE AVEC SUCCÈS!")
print("="*70)
print("\nRésumé final:")
print(f"  • Dataset: {df.shape[0]} lignes, {df.shape[1]} colonnes")
print(f"  • Après nettoyage: {df_clean.shape[0]} lignes, {df_clean.shape[1]} colonnes")
print(f"  • Features analysées: {len(X.columns)}")
print(f"  • Visualisations générées: 12+ graphiques")
print("\nL'analyse a identifié les facteurs clés influençant")
print("   la classification des prix des téléphones mobiles.")